# 04 - Model Evaluation
RMSE, MAE, F1, SSI, Wasserstein Distance

**Inputs (from Notebook 03):** `convlstm_model.keras`, `X_test.npy`, `y_test.npy`

**Outputs:** `predictions.npy`, `evaluation_results.csv`

In [1]:
!pip install -q tensorflow scikit-learn scipy

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score
from scipy.stats import wasserstein_distance
import json, os
# CELL: imports, drive mount, and shared pipeline CONFIG
from google.colab import drive

drive.mount('/content/drive')

CONFIG = {
    # ------------------------------------------------------------------
    # Spatial
    # ------------------------------------------------------------------
    # Broad staging bbox (Notebook 01 raw cache)
    'bbox_regional': {
        'lat_min': 0,   'lat_max': 30,
        'lon_min': 110, 'lon_max': 140,
    },
    # WPS model bbox (Notebook 02 onward)
    'bbox_model': {
        'lat_min': 10,  'lat_max': 20,
        'lon_min': 114, 'lon_max': 120,
    },

    # ------------------------------------------------------------------
    # Temporal
    # ------------------------------------------------------------------
    'date_full':  {'start': '2014-01-01', 'end': '2024-12-31'},
    'date_model': {'start': '2019-01-01', 'end': '2024-12-31'},

    # ------------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------------
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        # Source CMEMS NetCDF files placed in data_dir by the user
        'physics_w_nc':  'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc': 'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':    'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        # 01 outputs -> 02 inputs
        'physics_nc':    'physics_raw_region.nc',
        'bgc_nc':        'bgc_raw_region.nc',
        'ais_parquet':   'ais_raw_region.parquet',
        'ais_csv_gz':    'ais_raw_region.csv.gz',
        # 02 outputs -> 03 inputs
        'ais_gridded_nc':  'ais_fishing_effort_gridded.nc',
        'preprocessed_nc': 'preprocessed_features.nc',
        # 03 outputs -> 04 / 05 / dashboard inputs
        'model_keras':     'convlstm_model.keras',
        'best_model':      'best_model.keras',
        'X_test_npy':      'X_test.npy',
        'y_test_npy':      'y_test.npy',
        'history_json':    'training_history.json',
        'summary_json':    'data_summary.json',
        # 04 outputs -> 05 / dashboard inputs
        'predictions_npy': 'predictions.npy',
        'eval_csv':        'evaluation_results.csv',
    },

    # ------------------------------------------------------------------
    # AIS
    # ------------------------------------------------------------------
    'ais_use_cols': ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],

    # ------------------------------------------------------------------
    # Depth selection for CMEMS variables
    # ------------------------------------------------------------------
    'physics_surface_depth': 0.49,   # metres, nearest-neighbour selection
    'bgc_depth_range': (0.51, 5.14), # metres, averaged over this range

    # ------------------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------------------
    'norm_method':   'minmax',
    'resample_freq': '1ME',

    # ------------------------------------------------------------------
    # Model / Training
    # ------------------------------------------------------------------
    'seq_len':    3,     # months of input context fed to ConvLSTM
    'pred_len':   1,     # months ahead to predict
    'n_channels': 7,     # sst, ssh, vo, uo, chl, nppv, fishing_effort
    'train_frac': 0.70,
    'val_frac':   0.15,
    # test_frac = 1 - 0.70 - 0.15 = 0.15
    'epochs':     50,
    'batch_size': 8,
    'patience':   10,

    # ------------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------------
    'f1_threshold': 0.15,   # binarization threshold for F1 score in 04_evaluation
}

DATA_DIR   = CONFIG['data_dir']
f          = CONFIG['files']


Mounted at /content/drive


In [3]:
# CELL: load model and test arrays saved by Notebook 03
# Tries native .keras format first; falls back to legacy .h5 if not found.
model_keras_path = DATA_DIR + f['model_keras']
model_h5_fallback = DATA_DIR + 'convlstm_model.h5'

if os.path.exists(model_keras_path):
    model = tf.keras.models.load_model(model_keras_path)
    print(f'Loaded model  : {f["model_keras"]}')
elif os.path.exists(model_h5_fallback):
    model = tf.keras.models.load_model(model_h5_fallback)
    print(f'Loaded model  : convlstm_model.h5 (legacy fallback)')
else:
    raise FileNotFoundError(
        f'No model found at {model_keras_path} or {model_h5_fallback}. '
        'Run Notebook 03 first.'
    )

X_test = np.load(DATA_DIR + f['X_test_npy'])
y_test = np.load(DATA_DIR + f['y_test_npy'])

print(f'X_test shape  : {X_test.shape}')   # Expected: (N, 3, 41, 25, 7)
print(f'y_test shape  : {y_test.shape}')   # Expected: (N, 41, 25, 1)


Loaded model  : convlstm_model.keras
X_test shape  : (11, 3, 41, 25, 7)
y_test shape  : (11, 41, 25, 1)


In [4]:
# CELL: generate predictions and save to predictions.npy
# Fill NaN inputs (land/coastal cells) with 0 before predicting.
X_test_clean = np.nan_to_num(X_test, nan=0.0)
preds        = model.predict(X_test_clean)

print(f'Predictions shape: {preds.shape}')   # Expected: (N, 41, 25, 1)

# Save predictions for 05_visualization.ipynb and the dashboard
np.save(DATA_DIR + f['predictions_npy'], preds)
print(f'Saved predictions -> {f["predictions_npy"]}')

# Flatten for global scalar metrics, cleaning any residual NaNs
y_flat = np.nan_to_num(y_test.flatten(),  nan=0.0)
p_flat = np.nan_to_num(preds.flatten(),   nan=0.0)
print(f'NaN in y_flat : {np.isnan(y_flat).sum()}')
print(f'NaN in p_flat : {np.isnan(p_flat).sum()}')


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Predictions shape: (11, 41, 25, 1)
Saved predictions -> predictions.npy
NaN in y_flat : 0
NaN in p_flat : 0


In [5]:
# CELL: RMSE and MAE (global scalars over all test pixels and months)
rmse = float(np.sqrt(mean_squared_error(y_flat, p_flat)))
mae  = float(mean_absolute_error(y_flat, p_flat))
print(f'RMSE : {rmse:.6f}')
print(f'MAE  : {mae:.6f}')


RMSE : 0.206830
MAE  : 0.147374


In [6]:
# CELL: F1 score with threshold sweep (finds optimal threshold)
# Also reports F1 at the CONFIG default threshold for comparison.

thresholds_to_test = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
f1_scores = []

for thresh in thresholds_to_test:
    y_binary    = (y_flat > thresh).astype(int)
    pred_binary = (p_flat > thresh).astype(int)
    f1 = f1_score(y_binary, pred_binary, zero_division=0)
    f1_scores.append(f1)
    print(f'Threshold {thresh:.2f} -> F1 = {f1:.4f}')

# Find best threshold
best_idx = int(np.argmax(f1_scores))
best_threshold = thresholds_to_test[best_idx]
best_f1 = f1_scores[best_idx]
print(f'\n** Best F1: {best_f1:.4f} at threshold {best_threshold:.2f} **')

# Also compute F1 at CONFIG default for comparison
config_threshold = CONFIG['f1_threshold']
y_binary_cfg    = (y_flat > config_threshold).astype(int)
pred_binary_cfg = (p_flat > config_threshold).astype(int)
f1_config = float(f1_score(y_binary_cfg, pred_binary_cfg, zero_division=0))
print(f'F1 at CONFIG threshold ({config_threshold}): {f1_config:.4f}')

# Use best F1 for final results table
f1 = best_f1

Threshold 0.05 -> F1 = 0.5058
Threshold 0.10 -> F1 = 0.6920
Threshold 0.15 -> F1 = 0.0065
Threshold 0.20 -> F1 = 0.0000
Threshold 0.25 -> F1 = 0.0000
Threshold 0.30 -> F1 = 0.0000
Threshold 0.40 -> F1 = 0.0000
Threshold 0.50 -> F1 = 0.0000

** Best F1: 0.6920 at threshold 0.10 **
F1 at CONFIG threshold (0.15): 0.0065


In [7]:
# CELL: SSI (Structural Similarity Index) -- per-sample, then averaged
def calculate_ssi(obs, pred):
    C1, C2 = 0.01**2, 0.03**2
    mu_obs, mu_pred       = obs.mean(), pred.mean()
    sigma_obs, sigma_pred = obs.std(),  pred.std()
    sigma_cross = np.mean((obs - mu_obs) * (pred - mu_pred))
    luminance = (2*mu_obs*mu_pred + C1) / (mu_obs**2 + mu_pred**2 + C1)
    contrast  = (2*sigma_obs*sigma_pred + C2) / (sigma_obs**2 + sigma_pred**2 + C2)
    structure = (sigma_cross + C2/2) / (sigma_obs*sigma_pred + C2/2)
    return luminance * contrast * structure

ssi_scores = []
for i in range(len(y_test)):
    yi = y_test[i].flatten()
    pi = preds[i].flatten()
    m  = ~(np.isnan(yi) | np.isnan(pi))
    ssi_scores.append(calculate_ssi(yi[m], pi[m]))

ssi_mean = float(np.mean(ssi_scores))
ssi_std  = float(np.std(ssi_scores))
print(f'SSI : {ssi_mean:.4f} +/- {ssi_std:.4f}')


SSI : 0.1442 +/- 0.0237


In [8]:
# CELL: Wasserstein Distance -- per-sample distribution comparison
wd_scores = []
for i in range(len(y_test)):
    obs_flat  = y_test[i].flatten()
    pred_flat = preds[i].flatten()
    m = ~(np.isnan(obs_flat) | np.isnan(pred_flat))
    obs_flat, pred_flat = obs_flat[m], pred_flat[m]
    obs_sum  = obs_flat.sum()
    pred_sum = pred_flat.sum()
    if obs_sum == 0 or pred_sum == 0:
        wd_scores.append(np.nan)
        continue
    wd = wasserstein_distance(
        np.arange(len(obs_flat)),
        np.arange(len(pred_flat)),
        obs_flat / obs_sum,
        pred_flat / pred_sum,
    )
    wd_scores.append(wd)

wd_valid = [s for s in wd_scores if not np.isnan(s)]
wd_mean  = float(np.mean(wd_valid))
wd_std   = float(np.std(wd_valid))
print(f'Wasserstein Distance : {wd_mean:.4f} +/- {wd_std:.4f}')


Wasserstein Distance : 111.5479 +/- 35.0917


In [9]:
# CELL: compile results table and save to evaluation_results.csv
results = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'F1_best', 'F1_config', 'SSI', 'Wasserstein'],
    'Value': [
        f'{rmse:.3e}',
        f'{mae:.3e}',
        f'{best_f1:.4f}',
        f'{f1_config:.4f}',
        f'{ssi_mean:.4f}',
        f'{wd_mean:.4f}',
    ],
    'Std Dev': ['N/A', 'N/A', f'(thresh={best_threshold})', f'(thresh={config_threshold})', f'{ssi_std:.4f}', f'{wd_std:.4f}'],
})

print('=' * 50)
print('EVALUATION RESULTS')
print('=' * 50)
print(results.to_string(index=False))

results.to_csv(DATA_DIR + f['eval_csv'], index=False)
print(f'\nSaved evaluation results -> {f["eval_csv"]}')
print('Notebook 04 complete. Run 05_visualization.ipynb next.')

EVALUATION RESULTS
     Metric     Value       Std Dev
       RMSE 2.068e-01           N/A
        MAE 1.474e-01           N/A
    F1_best    0.6920  (thresh=0.1)
  F1_config    0.0065 (thresh=0.15)
        SSI    0.1442        0.0237
Wasserstein  111.5479       35.0917

Saved evaluation results -> evaluation_results.csv
Notebook 04 complete. Run 05_visualization.ipynb next.
